# Stream Media API Demo

This notebook demonstrates the full API surface of the **stream-media** project — a Rust microservices platform for managing and streaming media files with user management.

## Architecture

| Service | Port | Description |
|---|---|---|
| **Gateway** | 3000 | Reverse proxy — single entry point for all API calls |
| **Catalog Service** | 3001 | Media metadata CRUD (SQLite) |
| **Streaming Service** | 3002 | File upload and HTTP range streaming |
| **User Service** | 3003 | User management CRUD (SQLite) |

## Prerequisites

Start all services before running this notebook:

```bash
# Option 1: Docker Compose
docker compose up -d

# Option 2: Run each service locally (in separate terminals)
cargo run -p catalog-service
cargo run -p streaming-service
cargo run -p user-service
cargo run -p gateway
```

In [ ]:
import requests
import json

BASE_URL = "http://localhost:3000"

def pp(resp):
    """Pretty-print an HTTP response."""
    print(f"{resp.request.method} {resp.request.url}")
    print(f"Status: {resp.status_code}")
    if resp.headers.get("content-type", "").startswith("application/json"):
        print(json.dumps(resp.json(), indent=2))
    elif resp.text:
        print(resp.text[:500])
    print()

---

## 1. User Management

The User Service provides CRUD operations for managing users. All requests go through the gateway at `/api/users`.

### 1.1 Create Users

In [ ]:
# Create a few users
users_to_create = [
    {
        "username": "jpareja",
        "email": "jpareja@example.com",
        "display_name": "Joel Pareja"
    },
    {
        "username": "asmith",
        "email": "asmith@example.com",
        "display_name": "Alice Smith"
    },
    {
        "username": "bwilson",
        "email": "bwilson@example.com"
        # display_name is optional
    },
]

created_users = []
for user in users_to_create:
    resp = requests.post(f"{BASE_URL}/api/users", json=user)
    pp(resp)
    if resp.status_code == 201:
        created_users.append(resp.json())

### 1.2 List All Users

In [ ]:
# List all users (supports pagination with ?limit=&offset=)
resp = requests.get(f"{BASE_URL}/api/users")
pp(resp)

### 1.3 Search Users

Search across username, email, and display name.

In [ ]:
# Search users by keyword (matches username, email, or display_name)
resp = requests.get(f"{BASE_URL}/api/users", params={"search": "alice"})
pp(resp)

### 1.4 Get a Single User

In [ ]:
# Fetch a single user by ID
user_id = created_users[0]["id"]
resp = requests.get(f"{BASE_URL}/api/users/{user_id}")
pp(resp)

### 1.5 Update a User

Partial updates — only the fields you include will be changed.

In [ ]:
# Update user — only send fields you want to change
resp = requests.put(f"{BASE_URL}/api/users/{user_id}", json={
    "display_name": "Joel M. Pareja"
})
pp(resp)

### 1.6 Delete a User

In [ ]:
# Delete the third user
delete_id = created_users[2]["id"]
resp = requests.delete(f"{BASE_URL}/api/users/{delete_id}")
pp(resp)

# Verify deletion — should return 404
resp = requests.get(f"{BASE_URL}/api/users/{delete_id}")
pp(resp)

### 1.7 Uniqueness Constraints

Username and email must be unique. Duplicates return `400 Bad Request`.

In [ ]:
# Attempting to create a user with a duplicate username
resp = requests.post(f"{BASE_URL}/api/users", json={
    "username": "jpareja",          # already exists
    "email": "different@example.com"
})
pp(resp)  # 400 Bad Request

---

## 2. Media Catalog Management

The Catalog Service stores media metadata. You can create entries directly (metadata-only) or let the Streaming Service register them during upload.

### 2.1 Create Media Entries (Metadata Only)

In [ ]:
# Create media entries (metadata only — no file upload)
media_items = [
    {
        "title": "Bohemian Rhapsody",
        "description": "Queen's iconic rock opera from A Night at the Opera (1975)",
        "media_type": "audio",
        "format": "flac",
        "duration_secs": 354.0
    },
    {
        "title": "Comfortably Numb",
        "description": "Pink Floyd classic from The Wall (1979)",
        "media_type": "audio",
        "format": "mp3",
        "duration_secs": 382.0
    },
    {
        "title": "Nature Documentary - Coral Reefs",
        "description": "4K underwater footage of the Great Barrier Reef",
        "media_type": "video",
        "format": "mp4",
        "duration_secs": 2700.0
    },
]

created_media = []
for item in media_items:
    resp = requests.post(f"{BASE_URL}/api/media", json=item)
    pp(resp)
    if resp.status_code == 201:
        created_media.append(resp.json())

### 2.2 List All Media

In [ ]:
# List all media items
resp = requests.get(f"{BASE_URL}/api/media")
pp(resp)

### 2.3 Filter by Media Type

In [ ]:
# Filter by media type: "audio" or "video"
resp = requests.get(f"{BASE_URL}/api/media", params={"media_type": "audio"})
pp(resp)

### 2.4 Search Media by Title

In [ ]:
# Search by title (case-insensitive substring match)
resp = requests.get(f"{BASE_URL}/api/media", params={"search": "bohemian"})
pp(resp)

### 2.5 Pagination

In [ ]:
# Paginate results: first page (1 item)
resp = requests.get(f"{BASE_URL}/api/media", params={"limit": 1, "offset": 0})
pp(resp)

# Second page
resp = requests.get(f"{BASE_URL}/api/media", params={"limit": 1, "offset": 1})
pp(resp)

### 2.6 Get a Single Media Item

In [ ]:
# Fetch a single media item by ID
media_id = created_media[0]["id"]
resp = requests.get(f"{BASE_URL}/api/media/{media_id}")
pp(resp)

### 2.7 Update Media Metadata

In [ ]:
# Partial update — only title and description
resp = requests.put(f"{BASE_URL}/api/media/{media_id}", json={
    "title": "Bohemian Rhapsody (Remastered 2011)",
    "description": "Remastered version from the 2011 deluxe edition"
})
pp(resp)

### 2.8 Delete a Media Item

In [ ]:
# Delete media item
delete_media_id = created_media[1]["id"]
resp = requests.delete(f"{BASE_URL}/api/media/{delete_media_id}")
pp(resp)

# Confirm it's gone
resp = requests.get(f"{BASE_URL}/api/media/{delete_media_id}")
pp(resp)

---

## 3. File Upload and Streaming

The Streaming Service handles multipart file uploads and HTTP range-based streaming. Files are stored on disk and their metadata is registered in the catalog.

### 3.1 Upload a Media File

Upload uses `multipart/form-data` with metadata fields and a file.

In [ ]:
# Create a sample file for upload demo
sample_content = b"RIFF" + b"\x00" * 1020  # minimal fake WAV header (1KB)
sample_path = "/tmp/demo-sample.wav"
with open(sample_path, "wb") as f:
    f.write(sample_content)

# Upload via multipart form
# NOTE: the "format" field must appear BEFORE the "file" field
resp = requests.post(f"{BASE_URL}/upload", files={
    "title":       (None, "Demo Upload Track"),
    "description": (None, "A sample file uploaded via the API"),
    "media_type":  (None, "audio"),
    "format":      (None, "wav"),
    "file":        ("demo-sample.wav", open(sample_path, "rb"), "audio/wav"),
})
pp(resp)

if resp.status_code == 201:
    uploaded_item = resp.json()
    uploaded_id = uploaded_item["id"]

### 3.2 Stream the Full File

A simple GET request downloads the entire file.

In [ ]:
# Stream the full file (HTTP 200)
resp = requests.get(f"{BASE_URL}/stream/{uploaded_id}")
print(f"GET {resp.url}")
print(f"Status: {resp.status_code}")
print(f"Content-Type: {resp.headers.get('content-type')}")
print(f"Content-Length: {resp.headers.get('content-length')}")
print(f"Accept-Ranges: {resp.headers.get('accept-ranges')}")
print(f"Body size: {len(resp.content)} bytes")
print()

### 3.3 HTTP Range Requests

Range requests return `206 Partial Content` with a `Content-Range` header — essential for seeking in audio/video players.

In [ ]:
# Range request — fetch bytes 0 through 99 (first 100 bytes)
resp = requests.get(f"{BASE_URL}/stream/{uploaded_id}", headers={"Range": "bytes=0-99"})
print(f"Status: {resp.status_code}")  # 206 Partial Content
print(f"Content-Range: {resp.headers.get('content-range')}")
print(f"Content-Length: {resp.headers.get('content-length')}")
print(f"Body size: {len(resp.content)} bytes")
print()

# Open-ended range — from byte 512 to end of file
resp = requests.get(f"{BASE_URL}/stream/{uploaded_id}", headers={"Range": "bytes=512-"})
print(f"Status: {resp.status_code}")
print(f"Content-Range: {resp.headers.get('content-range')}")
print(f"Content-Length: {resp.headers.get('content-length')}")
print(f"Body size: {len(resp.content)} bytes")
print()

# Suffix range — last 256 bytes
resp = requests.get(f"{BASE_URL}/stream/{uploaded_id}", headers={"Range": "bytes=-256"})
print(f"Status: {resp.status_code}")
print(f"Content-Range: {resp.headers.get('content-range')}")
print(f"Content-Length: {resp.headers.get('content-length')}")
print(f"Body size: {len(resp.content)} bytes")

---

## 4. API Reference

### Endpoints Summary

| Method | Path | Description |
|---|---|---|
| `GET` | `/api/users` | List users (query: `search`, `limit`, `offset`) |
| `POST` | `/api/users` | Create user |
| `GET` | `/api/users/{id}` | Get user by ID |
| `PUT` | `/api/users/{id}` | Update user (partial) |
| `DELETE` | `/api/users/{id}` | Delete user |
| `GET` | `/api/media` | List media (query: `search`, `media_type`, `limit`, `offset`) |
| `POST` | `/api/media` | Create media metadata |
| `GET` | `/api/media/{id}` | Get media by ID |
| `PUT` | `/api/media/{id}` | Update media (partial) |
| `DELETE` | `/api/media/{id}` | Delete media |
| `POST` | `/upload` | Upload media file (multipart) |
| `GET` | `/stream/{id}` | Stream media file (supports Range) |

### Sample Payloads

#### Create User
```json
{
    "username": "jpareja",
    "email": "jpareja@example.com",
    "display_name": "Joel Pareja"
}
```

#### Update User
```json
{
    "display_name": "Joel M. Pareja"
}
```

#### Create Media
```json
{
    "title": "Bohemian Rhapsody",
    "description": "Queen's iconic rock opera from A Night at the Opera (1975)",
    "media_type": "audio",
    "format": "flac",
    "duration_secs": 354.0
}
```

#### Update Media
```json
{
    "title": "Bohemian Rhapsody (Remastered 2011)",
    "description": "Remastered version from the 2011 deluxe edition"
}
```

#### Upload File (multipart/form-data)
```
title:        "My Track"
description:  "Optional description"
media_type:   "audio"          # "audio" or "video"
format:       "mp3"            # must appear before file field
file:         <binary data>
```

#### Streaming with Range Header
```
Range: bytes=0-1023       # first 1KB
Range: bytes=1048576-     # from 1MB to end
Range: bytes=-524288      # last 512KB
```

---

## 5. Cleanup

Remove the demo data created during this notebook run.

In [ ]:
# Delete all users created in this session
for user in created_users:
    resp = requests.delete(f"{BASE_URL}/api/users/{user['id']}")
    status = "deleted" if resp.status_code == 204 else f"skipped ({resp.status_code})"
    print(f"User {user.get('username', user['id'])}: {status}")

# Delete all media created in this session
for item in created_media:
    resp = requests.delete(f"{BASE_URL}/api/media/{item['id']}")
    status = "deleted" if resp.status_code == 204 else f"skipped ({resp.status_code})"
    print(f"Media {item.get('title', item['id'])}: {status}")

# Delete uploaded item
if 'uploaded_id' in dir():
    resp = requests.delete(f"{BASE_URL}/api/media/{uploaded_id}")
    status = "deleted" if resp.status_code == 204 else f"skipped ({resp.status_code})"
    print(f"Uploaded item: {status}")

print("\nCleanup complete.")